# 09 — 1D Convolution

## Background

Convolution is the backbone of CNNs and is widely used in audio processing and sequence modelling (WaveNet, Mamba's SSM convolution, etc.). Its defining feature is **weight sharing**: the same kernel slides and is reused across positions.

GPU implementation challenges:
1. **Boundary handling**: same padding requires zero-padding out-of-bounds positions — use branchless writes to avoid warp divergence
2. **Access pattern**: adjacent output positions overlap in their input window (sliding), so naive implementations repeatedly load the same data

## Problem Definition

**Single-channel 1D convolution (same padding)**:
$$O[n, l] = \sum_{k=0}^{KL-1} X[n,\ l + k - \text{pad}] \times K[k], \quad \text{pad} = \lfloor(KL-1)/2\rfloor$$

Out-of-bounds positions are treated as 0.

**Multi-output-channel 1D convolution**:
$$O[n, l, f] = \sum_{k=0}^{KL-1} X[n,\ l+k-\text{pad}] \times K[k, f]$$

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch Reference

In [ ]:
# Single-channel
N_b, L, KL = 64, 1024, 7
X_1d  = torch.randn(N_b, L, dtype=torch.float16, device="cuda")
Kern  = torch.randn(KL,     dtype=torch.float16, device="cuda")

# Multi-channel
N_mc, L_mc, KL_mc, F = 16, 256, 7, 32
X_mc    = torch.randn(N_mc, L_mc, dtype=torch.float16, device="cuda")
Kern_mc = torch.randn(KL_mc, F,   dtype=torch.float16, device="cuda")

def ref_conv1d(X, K):
    pad = (K.shape[0] - 1) // 2
    return torch.conv1d(X.unsqueeze(1), K.view(1, 1, -1), padding=pad).squeeze(1)

def ref_conv1d_mc(X, K):
    N_, L_ = X.shape;  KL_, F_ = K.shape;  pad = (KL_ - 1) // 2
    O = torch.zeros(N_, L_, F_, dtype=X.dtype, device=X.device)
    for f in range(F_):
        O[:, :, f] = torch.conv1d(X.unsqueeze(1), K[:, f].view(1, 1, -1), padding=pad).squeeze(1)
    return O

O1d_ref = ref_conv1d(X_1d, Kern)
Omc_ref = ref_conv1d_mc(X_mc, Kern_mc)
print(f"Single-ch: X{X_1d.shape} * K{Kern.shape} → O{O1d_ref.shape}")
print(f"Multi-ch:  X{X_mc.shape} * K{Kern_mc.shape} → O{Omc_ref.shape}")

## Triton Implementation

Each Triton program covers a `[BN, BL]` output tile:
- Outer loop over `KL` kernel taps; for each, compute `src = l + k - pad`
- Boundary mask `(src >= 0) & (src < L)` with `other=0.0` eliminates branching
- Multi-channel variant: add a third grid dimension for `F`; each program handles one fixed `f`

In [ ]:
@triton.jit
def _conv1d_kernel(X_ptr, K_ptr, O_ptr, N, L, KL, pad,
                   BN: tl.constexpr, BL: tl.constexpr):
    pid_n = tl.program_id(0);  pid_l = tl.program_id(1)
    ns  = pid_n * BN + tl.arange(0, BN)
    ls  = pid_l * BL + tl.arange(0, BL)
    acc = tl.zeros([BN, BL], dtype=tl.float32)
    for k in range(KL):
        src  = ls + k - pad
        mask = (ns[:, None] < N) & (src[None, :] >= 0) & (src[None, :] < L)
        x    = tl.load(X_ptr + ns[:, None] * L + src[None, :], mask=mask, other=0.0)
        kv   = tl.load(K_ptr + k)
        acc  = acc + x.to(tl.float32) * kv.to(tl.float32)
    mask_o = (ns[:, None] < N) & (ls[None, :] < L)
    tl.store(O_ptr + ns[:, None] * L + ls[None, :], acc.to(tl.float16), mask=mask_o)

@triton.jit
def _conv1d_mc_kernel(X_ptr, K_ptr, O_ptr, N, L, KL, F, pad,
                      BN: tl.constexpr, BL: tl.constexpr):
    pid_n = tl.program_id(0);  pid_l = tl.program_id(1);  f = tl.program_id(2)
    ns  = pid_n * BN + tl.arange(0, BN)
    ls  = pid_l * BL + tl.arange(0, BL)
    acc = tl.zeros([BN, BL], dtype=tl.float32)
    for k in range(KL):
        src  = ls + k - pad
        mask = (ns[:, None] < N) & (src[None, :] >= 0) & (src[None, :] < L)
        x    = tl.load(X_ptr + ns[:, None] * L + src[None, :], mask=mask, other=0.0)
        kv   = tl.load(K_ptr + k * F + f, mask=f < F, other=0.0)
        acc  = acc + x.to(tl.float32) * kv.to(tl.float32)
    mask_o = (ns[:, None] < N) & (ls[None, :] < L) & (f < F)
    tl.store(O_ptr + (ns[:, None] * L + ls[None, :]) * F + f,
             acc.to(tl.float16), mask=mask_o)

BN_t, BL_t = 4, 64

def triton_conv1d(X, K):
    N_, L_ = X.shape;  KL_ = K.shape[0];  pad = (KL_ - 1) // 2
    O = torch.empty_like(X)
    _conv1d_kernel[(triton.cdiv(N_, BN_t), triton.cdiv(L_, BL_t))](
        X, K, O, N_, L_, KL_, pad, BN=BN_t, BL=BL_t)
    return O

def triton_conv1d_mc(X, K):
    N_, L_ = X.shape;  KL_, F_ = K.shape;  pad = (KL_ - 1) // 2
    O = torch.empty(N_, L_, F_, dtype=X.dtype, device=X.device)
    _conv1d_mc_kernel[(triton.cdiv(N_, BN_t), triton.cdiv(L_, BL_t), F_)](
        X, K, O, N_, L_, KL_, F_, pad, BN=BN_t, BL=BL_t)
    return O

ok1 = torch.allclose(O1d_ref, triton_conv1d(X_1d, Kern), atol=1e-2)
ok2 = torch.allclose(Omc_ref, triton_conv1d_mc(X_mc, Kern_mc), atol=1e-2)
print("Triton single-ch:", "✅" if ok1 else "❌", "  multi-ch:", "✅" if ok2 else "❌")

## TileLang Implementation

`T.if_then_else((src >= 0) & (src < L), X[n, src], T.float16(0))` provides branchless zero padding — semantically equivalent to Triton's `mask + other=0.0`.

In [ ]:
@tilelang.jit
def tl_conv1d(X, K, BLOCK_N: int, BLOCK_L: int):
    N_, L_, KL_ = T.const("N, L, KL")
    dtype = T.float16
    X: T.Tensor((N_, L_), dtype);  K: T.Tensor((KL_,), dtype)
    O = T.empty((N_, L_), dtype)
    pad = (KL_ - 1) // 2
    with T.Kernel(N_ // BLOCK_N, L_ // BLOCK_L, threads=256) as (pid_n, pid_l):
        for i, j in T.Parallel(BLOCK_N, BLOCK_L):
            n = pid_n * BLOCK_N + i
            l = pid_l * BLOCK_L + j
            val = T.float16(0)
            for k in T.Serial(KL_):
                src = l + k - pad
                val = val + T.if_then_else(
                    (src >= 0) & (src < L_), X[n, src] * K[k], T.float16(0))
            O[n, l] = val
    return O

@tilelang.jit
def tl_conv1d_mc(X, K, BLOCK_N: int, BLOCK_L: int, BLOCK_F: int):
    N_, L_, KL_, F_ = T.const("N, L, KL, F")
    dtype = T.float16
    X: T.Tensor((N_, L_), dtype);  K: T.Tensor((KL_, F_), dtype)
    O = T.empty((N_, L_, F_), dtype)
    pad = (KL_ - 1) // 2
    with T.Kernel(N_ // BLOCK_N, L_ // BLOCK_L, F_ // BLOCK_F, threads=256) as (pid_n, pid_l, pid_f):
        for i, j, ff in T.Parallel(BLOCK_N, BLOCK_L, BLOCK_F):
            n = pid_n * BLOCK_N + i
            l = pid_l * BLOCK_L + j
            f = pid_f * BLOCK_F + ff
            val = T.float16(0)
            for k in T.Serial(KL_):
                src = l + k - pad
                val = val + T.if_then_else(
                    (src >= 0) & (src < L_), X[n, src] * K[k, f], T.float16(0))
            O[n, l, f] = val
    return O

k_1d = tl_conv1d.compile(N=N_b, L=L, KL=KL, BLOCK_N=4, BLOCK_L=64)
k_mc = tl_conv1d_mc.compile(N=N_mc, L=L_mc, KL=KL_mc, F=F, BLOCK_N=4, BLOCK_L=32, BLOCK_F=32)

ok1 = torch.allclose(O1d_ref, k_1d(X_1d.clone(), Kern.clone()), atol=1e-2)
ok2 = torch.allclose(Omc_ref, k_mc(X_mc.clone(), Kern_mc.clone()), atol=1e-2)
print("TileLang single-ch:", "✅" if ok1 else "❌", "  multi-ch:", "✅" if ok2 else "❌")

## Performance Comparison

In [ ]:
import matplotlib.pyplot as plt

WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

ms_1d = [bench(ref_conv1d,       [X_1d, Kern]),
         bench(triton_conv1d,    [X_1d, Kern]),
         bench(k_1d,             [X_1d, Kern])]
ms_mc = [bench(ref_conv1d_mc,    [X_mc, Kern_mc]),
         bench(triton_conv1d_mc, [X_mc, Kern_mc]),
         bench(k_mc,             [X_mc, Kern_mc])]

labels = ["PyTorch", "Triton", "TileLang"]
colors = ["#4C72B0", "#55A868", "#C44E52"]

print("Single-ch:", {l: f"{ms:.4f} ms" for l, ms in zip(labels, ms_1d)})
print("Multi-ch: ", {l: f"{ms:.4f} ms" for l, ms in zip(labels, ms_mc)})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, data, title in zip(axes, [ms_1d, ms_mc],
    [f"Single-channel Conv  (N={N_b}, L={L}, KL={KL})",
     f"Multi-channel Conv   (N={N_mc}, L={L_mc}, KL={KL_mc}, F={F})"]):
    bars = ax.bar(labels, data, color=colors, width=0.5, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, data):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(data)*0.01,
                f"{val:.4f} ms", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("Latency (ms)"); ax.set_title(title)
    ax.set_ylim(0, max(data)*1.3); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()